# ActiSense v5 – optimisation on Kaggle

**Goal:** better and more honest predictions, using public data only.

What changes against v4:
1. **No laptop recordings.** Standing, sitting and lying came from one person in one room (biased). In v5 they are
   learned by the *posture model* from public images of many people (Human Action Recognition dataset: sitting,
   using a laptop, sleeping) and from KTH frames of 25 people standing. The activity model keeps only *moving*
   activities from KTH and Penn Action.
2. **Activity model:** joint velocity features, mirror and speed augmentation, and three models compared
   (LSTM, BiLSTM with attention, temporal CNN). Every setting is one row of `ablation.csv`; the final setting is
   chosen on the **validation** set and retrained with 3 seeds.
3. **Posture model:** extra angle features (torso and thigh tilt, ankles) and camera-roll augmentation.

**Settings:** Internet **On**. Accelerator **GPU T4 x1** makes training about 10x faster (CPU works too).
**Inputs (Add Input):** `actisense-data` (private), `penn-action-dataset`, `yoga-poses-dataset`,
`bharatnatyam-dance-poses`, **`human-action-recognition-har-dataset`** (meetnagadia, new), and optionally the
**output of your v4 notebook run** (contains `extracted_features.zip` and `poses_*.npz`, so extraction is skipped).

Run the cells **one by one**, top to bottom. Then download `outputs/` from the Output tab.

In [ ]:
# 1. Code + a Python 3.11 environment for MediaPipe extraction (training uses Kaggle's own Python + GPU)
import os
os.environ['MPLBACKEND'] = 'Agg'
%cd /kaggle/working
!rm -rf actisense && git clone -q https://github.com/SHARMI-P/actisense.git && git -C actisense log --oneline -1
!pip install -q uv
!uv venv -q --clear -p 3.11 /tmp/py311
!uv pip install -q --python /tmp/py311/bin/python "mediapipe==0.10.14" "numpy<2" opencv-contrib-python scikit-learn matplotlib pandas scipy
!uv pip install -q --python /tmp/py311/bin/python torch --index-url https://download.pytorch.org/whl/cpu || uv pip install -q --python /tmp/py311/bin/python torch
!/tmp/py311/bin/python -c "import sys, mediapipe as mp; print('extraction env: python', sys.version.split()[0], '| mediapipe', mp.__version__)"
!python -c "import torch; print('training env: torch', torch.__version__, '| GPU:', torch.cuda.is_available())"


In [ ]:
# 2. Locate the attached datasets (searched a few folder levels deep, wherever Kaggle mounts them)
import glob
I = '/kaggle/input'
def find(tail, required=True):
    for depth in range(0, 7):
        hits = sorted(glob.glob(os.path.join(I, *['*'] * depth, tail)))
        if hits:
            return hits[0]
    if required:
        raise SystemExit(f'Not found: {tail} - add the dataset with Add Input')
    return None
ACTI   = find('actisense-data')
PENN   = find('Penn_Action', required=False)
YOGA   = find('yoga-poses-dataset/DATASET')
BHARAT = find('new_Dataset')
HARCSV = find('Training_set.csv')                                  # Human Action Recognition dataset
HARIMG = os.path.join(os.path.dirname(HARCSV), 'train')
PREV_ZIP = find('extracted_features.zip', required=False)          # windows from the v4 run (optional)
PREV_DIR = os.path.dirname(PREV_ZIP) if PREV_ZIP else None
for k, v in dict(ACTI=ACTI, PENN=PENN, YOGA=YOGA, BHARAT=BHARAT, HARCSV=HARCSV, PREV_ZIP=PREV_ZIP).items():
    print(f'{k:9s}', v)
print('activity folders in actisense-data:', sorted(os.listdir(ACTI)))
PY = '/tmp/py311/bin/python'          # MediaPipe extraction
os.environ['GLOG_minloglevel'] = '2'
TEST = 'person(02|03|05|06|07|08|09|10|22)_|pennTest_'   # KTH official test people + Penn official test videos


In [ ]:
assert 'PY' in globals() and os.path.exists(PY), 'Session was restarted: run cells 1 and 2 first, then this cell.'
# 3. Penn Action windows: reuse the v4 extraction if attached, else extract (20-60 min)
if PREV_ZIP:
    !rm -rf data_penn && unzip -q -o "{PREV_ZIP}" -d . && echo "reused data_penn from the v4 run"
else:
    assert PENN, 'Attach penn-action-dataset (or the v4 notebook output with extracted_features.zip)'
    !{PY} actisense/extract_penn.py --root "{PENN}" --out data_penn --workers 4 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'
for d in sorted(os.listdir('data_penn')):
    print(f'{d:16s}', len(os.listdir(os.path.join('data_penn', d))))


In [ ]:
assert 'PY' in globals() and os.path.exists(PY), 'Session was restarted: run cells 1 and 2 first, then this cell.'
# 4. Posture images -> skeletons: yoga, Bharatanatyam (reused if attached) and NEW: sitting / lying images
for name, root, extra in [('poses_yoga_train.npz', f'{YOGA}/TRAIN', '--split train --source yoga'),
                          ('poses_yoga_test.npz',  f'{YOGA}/TEST',  '--split test --source yoga'),
                          ('poses_bharat.npz',     BHARAT,          '--strip " Augmented" --source bharatanatyam')]:
    prev = os.path.join(PREV_DIR, name) if PREV_DIR else ''
    if prev and os.path.exists(prev):
        !cp "{prev}" . && echo "reused {name}"
    else:
        !{PY} actisense/extract_images.py --root "{root}" {extra} --out {name} 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'
# Human Action Recognition dataset: many different people; using_laptop is sitting at a computer, sleeping = lying
!{PY} actisense/extract_images.py --root "{HARIMG}" --csv "{HARCSV}" --map using_laptop=sitting,sleeping=lying --only sitting,lying --source har --out poses_har.npz 2>&1 | grep --line-buffered -v -E 'W0000|I0000|INFO:|WARNING: All log|feedback tensors|UserWarning|warnings.warn'


In [ ]:
# 5. Activity model: ablation study (one change at a time), all on the same 16 moving activities
#    and the same honest test set; own recordings left out, standing/sitting/lying handled by the posture model
import torch
SEEDS = 3 if torch.cuda.is_available() else 1          # on CPU: 1 seed per row to save time
BASE = f'--data "{ACTI}" data_penn --test-pattern "{TEST}" --features xyz --exclude-own --drop-classes standing,sitting,lying --seeds {SEEDS}'
runs = [('a1_v4settings', '--augment --model lstm'),
        ('a2_velocity',   '--augment --velocity --model lstm'),
        ('a3_augment',    '--augment --velocity --mirror --speed --model lstm'),
        ('a4_bilstm',     '--augment --velocity --mirror --speed --model bilstm'),
        ('a5_tcn',        '--augment --velocity --mirror --speed --model tcn')]
!rm -f ablation.csv
for tag, opts in runs:
    print(f'\n########## {tag}: {opts}')
    !python actisense/train.py {BASE} {opts} --tag {tag} 2>&1 | grep -E "Classes|Split|^seed|Test |Accuracy on|Saved"
import pandas as pd
abl = pd.read_csv('ablation.csv')
abl[['tag', 'model', 'velocity', 'mirror', 'speed', 'val_macro_f1', 'accuracy', 'macro_f1', 'macro_f1_sd', 'video_accuracy']]


In [ ]:
# 6. Final activity model: the setting with the best VALIDATION macro-F1, retrained with 3 seeds
abl = pd.read_csv('ablation.csv')
best = abl.loc[abl['val_macro_f1'].idxmax()]
opts = dict(runs)[best['tag']]
print('Chosen on validation:', best['tag'], '->', opts)
FINAL = BASE.replace(f'--seeds {SEEDS}', '--seeds 3')
!python actisense/train.py {FINAL} {opts} --tag v5 2>&1 | grep -v "  seed"


In [ ]:
# 7. Posture model v5: yoga + Bharatanatyam + standing / sitting / lying from public data + 'none'
!python actisense/train_pose.py --poses poses_yoga_train.npz poses_yoga_test.npz poses_bharat.npz poses_har.npz \
    --standing-from "{ACTI}" --standing-classes boxing,clapping,waving --standing-per-class 400 \
    --negatives data_penn --negative-classes pushup,situp,pullup,jump_rope,jumping_jacks,squatting \
    --test-pattern "{TEST}" --extended --tag v5 2>&1 | grep -v "^epoch"


In [ ]:
# 8. Collect everything to download (Output tab -> outputs/)
import shutil
from IPython.display import Image, display
os.makedirs('outputs', exist_ok=True)
shutil.copy('har_model_v5.pt', 'outputs/har_model.pt')        # the live app loads these two names
shutil.copy('pose_model_v5.pt', 'outputs/pose_model.pt')
for f in ['results_v5.txt', 'results_pose_v5.txt', 'confusion_matrix_v5.png', 'confusion_matrix_pose_v5.png',
          'ablation.csv'] + glob.glob('results_a*.txt'):
    shutil.copy(f, 'outputs/')
shutil.make_archive('outputs/extracted_features', 'zip', '.', 'data_penn')   # reuse next time
for f in glob.glob('poses_*.npz'):
    shutil.copy(f, 'outputs/')
print(open('outputs/results_v5.txt').read())
print(open('outputs/results_pose_v5.txt').read())
display(pd.read_csv('outputs/ablation.csv'))
display(Image('outputs/confusion_matrix_v5.png'))
display(Image('outputs/confusion_matrix_pose_v5.png'))
print(sorted(os.listdir('outputs')))
